<a href="https://colab.research.google.com/github/Desire-in-tech/Customer-Segmentation/blob/main/01_exploring_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: Exploring the Data

**Dataset:** Survey of Consumer Finances (SCF) 2019  
**Focus:** Households that were turned down for credit or feared being denied credit.

In this notebook we:
- Load the SCF 2019 dataset
- Subset to households with `TURNFEAR == 1`
- Explore demographics, income, assets, and debt
- Compute correlations and visualize distributions

In [1]:
REPO_URL = 'https://github.com/Desire-in-tech/Customer-Segmentation.git'

!git clone {REPO_URL}

fatal: destination path 'Customer-Segmentation' already exists and is not an empty directory.


In [2]:
import os

os.chdir('/content/Customer-Segmentation')

# Install required packages
!pip install pandas numpy matplotlib seaborn plotly scikit-learn scipy --quiet

print(f'Working directory: {os.getcwd()}')
print(f'Data file exists: {os.path.exists("data/SCF_2019.csv.gz")}')

Working directory: /content/Customer-Segmentation
Data file exists: True


## 1. Imports

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '${:,.2f}'.format)
pd.set_option('display.max_columns', 50)

print('Libraries loaded successfully!')

Libraries loaded successfully!


## 2. Load the Dataset

> The SCF uses **multiple imputation** (5 implicates per household). We keep every 5th row (implicate 1) to avoid duplication.

In [4]:
# Load the compressed dataset
df = pd.read_csv('data/SCF_2019.csv.gz', compression='gzip', index_col=0)

df = df.iloc[::5].reset_index(drop=True)

print(f'Dataset shape: {df.shape}')
print(f'Number of households: {len(df):,}')
df.head()

Dataset shape: (5777, 357)
Number of households: 5,777


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,LF,LIFECL,FAMSTRUCT,RACECL,RACECL4,RACECL5,RACECL_EX,RACE,OCCAT1,OCCAT2,INDCAT,FOODHOME,FOODAWAY,FOODDELV,RENT,...,PLOAN6,PLOAN7,PLOAN8,LLOAN1,LLOAN2,LLOAN3,LLOAN4,LLOAN5,LLOAN6,LLOAN7,LLOAN8,LLOAN9,LLOAN10,LLOAN11,LLOAN12,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
0,1,11,"$6,119.78",2,75,6,12,4,2,0,1,5,3,1,1,1,99,1,1,1,2,"$12,055.74","$3,477.62",$0.00,$0.00,...,$0.00,$0.00,0,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,5,3,6,3,2,10,6,6,3,3
1,2,21,"$3,790.48",1,50,3,8,2,1,3,1,3,4,1,1,1,99,1,4,4,4,"$6,677.02",$834.63,$0.00,"$1,275.13",...,$0.00,$0.00,0,$0.00,$0.00,"$14,142.30",$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,1,2,1,2,1,1,4,4,2,2
2,3,31,"$4,099.48",1,53,3,8,2,1,0,1,2,5,1,1,1,99,1,1,3,1,"$4,173.14","$4,173.14",$0.00,"$1,159.21",...,$0.00,$0.00,0,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$289.80,$0.00,$0.00,2,4,2,4,2,4,8,7,3,3
3,4,41,"$6,375.98",1,29,1,13,4,1,0,1,2,5,1,1,1,99,1,1,1,2,"$5,564.19","$2,503.88",$0.00,$0.00,...,"$205,179.34",$0.00,0,"$245,635.60",$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,"$205,179.34",$0.00,$0.00,$0.00,$0.00,1,4,3,4,2,1,8,8,4,4
4,5,51,"$4,210.23",1,47,3,8,2,1,0,1,2,5,2,4,5,99,1,1,2,2,"$5,285.98",$0.00,$0.00,$579.60,...,$0.00,$0.00,0,$0.00,$0.00,$0.00,$0.00,$0.00,$788.26,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,1,2,1,2,1,2,3,3,1,2


## 3. Subset: Households Turned Down or Fearing Denial of Credit

In [5]:
# Create mask for households turned down or fearing credit denial
mask = df['TURNFEAR'] == 1

# Create subset dataframe
df_fear = df[mask].copy().reset_index(drop=True)

print(f'Total households:              {len(df):,}')
print(f'TURNFEAR == 1 households:      {len(df_fear):,}')
print(f'Percentage of total:           {len(df_fear)/len(df)*100:.1f}%')

df_fear.head()

Total households:              5,777
TURNFEAR == 1 households:      926
Percentage of total:           16.0%


,YY1,Y1,WGT,HHSEX,AGE,AGECL,EDUC,EDCL,MARRIED,KIDS,LF,LIFECL,FAMSTRUCT,RACECL,RACECL4,RACECL5,RACECL_EX,RACE,OCCAT1,OCCAT2,INDCAT,FOODHOME,FOODAWAY,FOODDELV,RENT,...,PLOAN6,PLOAN7,PLOAN8,LLOAN1,LLOAN2,LLOAN3,LLOAN4,LLOAN5,LLOAN6,LLOAN7,LLOAN8,LLOAN9,LLOAN10,LLOAN11,LLOAN12,NWCAT,INCCAT,ASSETCAT,NINCCAT,NINC2CAT,NWPCTLECAT,INCPCTLECAT,NINCPCTLECAT,INCQRTCAT,NINCQRTCAT
0,2,21,"$3,790.48",1,50,3,8,2,1,3,1,3,4,1,1,1,99,1,4,4,4,"$6,677.02",$834.63,$0.00,"$1,275.13",...,$0.00,$0.00,0,$0.00,$0.00,"$14,142.30",$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,1,2,1,2,1,1,4,4,2,2
1,23,231,"$5,424.99",1,69,5,11,3,2,0,0,6,3,1,1,1,99,1,3,4,4,"$1,669.26",$0.00,$0.00,"$1,031.69",...,$0.00,$0.00,0,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,1,1,1,2,1,3,2,3,1,1
2,31,311,"$4,730.10",1,31,1,9,3,1,2,1,3,4,1,1,1,99,1,1,1,2,"$6,955.23","$2,086.57",$0.00,"$1,506.97",...,$0.00,$0.00,0,$0.00,$0.00,"$69,204.56",$0.00,$0.00,$0.00,"$1,159.21",$0.00,$0.00,"$3,129.85",$0.00,$0.00,1,3,2,3,2,1,6,6,3,3
3,37,371,"$4,705.03",1,45,3,2,1,1,0,1,2,5,2,3,3,99,3,4,4,4,"$9,041.80","$2,109.75",$0.00,$985.32,...,$0.00,$0.00,0,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,$0.00,1,2,1,2,1,3,4,4,2,2
4,40,401,"$5,620.47",1,68,5,12,4,2,0,0,6,3,2,2,2,99,2,3,4,4,"$4,173.14",$278.21,$0.00,$0.00,...,$0.00,$0.00,0,$0.00,$0.00,$0.00,$0.00,"$112,442.91",$0.00,$0.00,$0.00,$0.00,$324.58,$0.00,$0.00,2,2,3,2,1,5,4,3,2,2


## 4. Summary Statistics

In [6]:
# Key financial columns to explore
key_cols = ['INCOME', 'NETWORTH', 'ASSET', 'DEBT', 'HOUSES', 'MRTHEL',
            'CCBAL', 'SAVBND', 'RETQLIQ', 'NHNFIN', 'AGE', 'EDUC']

# Summary statistics
summary = df_fear[key_cols].describe().T
summary['median'] = df_fear[key_cols].median()
summary = summary[['count', 'mean', 'median', 'std', 'min', 'max']]
summary.columns = ['Count', 'Mean', 'Median', 'Std Dev', 'Min', 'Max']
summary

,Count,Mean,Median,Std Dev,Min,Max
INCOME,$926.00,"$128,916.25","$47,209.90","$832,200.96",$0.00,"$19,001,984.70"
NETWORTH,$926.00,"$1,594,186.09","$13,875.69","$19,444,015.88","$-1,029,420.67","$532,174,923.44"
ASSET,$926.00,"$1,740,338.69","$38,341.87","$19,810,532.80",$0.00,"$538,141,353.11"
DEBT,$926.00,"$146,152.59","$21,456.89","$796,521.39",$0.00,"$21,113,533.46"
HOUSES,$926.00,"$217,868.33",$0.00,"$1,514,519.74",$0.00,"$37,175,713.91"
MRTHEL,$926.00,"$95,816.20",$0.00,"$745,551.77",$0.00,"$20,865,695.36"
CCBAL,$926.00,"$4,913.21",$179.68,"$14,364.97",$0.00,"$176,199.21"
SAVBND,$926.00,$124.85,$0.00,"$1,671.88",$0.00,"$42,890.60"
RETQLIQ,$926.00,"$60,919.69",$0.00,"$389,900.70",$0.00,"$7,027,102.52"
NHNFIN,$926.00,"$1,363,114.76","$13,437.57","$19,680,011.37",$0.00,"$553,103,787.05"


## 5. Age Distribution

In [7]:
# Age series
age_series = df_fear['AGE']

fig = px.histogram(
    df_fear,
    x='AGE',
    nbins=30,
    title='Age Distribution of Households Turned Down for Credit',
    labels={'AGE': 'Age of Household Head', 'count': 'Number of Households'},
    color_discrete_sequence=['#636EFA']
)
fig.add_vline(
    x=age_series.median(),
    line_dash='dash',
    line_color='red',
    annotation_text=f'Median: {age_series.median():.0f}'
)
fig.update_layout(bargap=0.05)
fig.show()

print(f'Mean age:   {age_series.mean():.1f} years')
print(f'Median age: {age_series.median():.1f} years')
print(f'Std dev:    {age_series.std():.1f} years')

Mean age:   45.1 years
Median age: 44.0 years
Std dev:    14.9 years


## 6. Income Distribution

In [8]:
# Income series — filter extreme outliers for visualization
income_series = df_fear['INCOME']
income_trimmed = income_series[income_series < income_series.quantile(0.99)]

fig = px.histogram(
    x=income_trimmed,
    nbins=50,
    title='Income Distribution (households turned down for credit, excluding top 1%)',
    labels={'x': 'Annual Household Income ($)', 'count': 'Number of Households'},
    color_discrete_sequence=['#EF553B']
)
fig.add_vline(
    x=income_series.median(),
    line_dash='dash',
    line_color='navy',
    annotation_text=f'Median: ${income_series.median():,.0f}'
)
fig.update_layout(bargap=0.05)
fig.show()

print(f'Mean income:   ${income_series.mean():,.0f}')
print(f'Median income: ${income_series.median():,.0f}')
print(f'Std dev:       ${income_series.std():,.0f}')

Mean income:   $128,916
Median income: $47,210
Std dev:       $832,201


## 7. Net Worth Distribution

In [9]:
# Net worth series
networth_series = df_fear['NETWORTH']
networth_trimmed = networth_series[
    (networth_series > networth_series.quantile(0.01)) &
    (networth_series < networth_series.quantile(0.99))
]

fig = px.histogram(
    x=networth_trimmed,
    nbins=50,
    title='Net Worth Distribution (excluding bottom/top 1%)',
    labels={'x': 'Net Worth ($)', 'count': 'Number of Households'},
    color_discrete_sequence=['#00CC96']
)
fig.add_vline(
    x=networth_series.median(),
    line_dash='dash',
    line_color='red',
    annotation_text=f'Median: ${networth_series.median():,.0f}'
)
fig.update_layout(bargap=0.05)
fig.show()

print(f'Mean net worth:   ${networth_series.mean():,.0f}')
print(f'Median net worth: ${networth_series.median():,.0f}')
print(f'Negative net worth (households): {(networth_series < 0).sum():,}')

Mean net worth:   $1,594,186
Median net worth: $13,876
Negative net worth (households): 233


## 8. Education Level Breakdown

In [10]:
# Education labels
educ_labels = {
    1: 'No High School',
    2: 'High School Diploma',
    3: 'Some College',
    4: 'College Degree',
    5: 'Graduate Degree'
}

educ_series = df_fear['EDUC'].map(educ_labels)
educ_counts = educ_series.value_counts().reset_index()
educ_counts.columns = ['Education Level', 'Count']

# Order by education level
educ_order = list(educ_labels.values())
educ_counts['Education Level'] = pd.Categorical(
    educ_counts['Education Level'], categories=educ_order, ordered=True
)
educ_counts = educ_counts.sort_values('Education Level')

fig = px.bar(
    educ_counts,
    x='Education Level',
    y='Count',
    title='Education Level of Households Turned Down for Credit',
    color='Count',
    color_continuous_scale='Blues'
)
fig.update_layout(xaxis_title='Education Level', yaxis_title='Number of Households')
fig.show()

print(educ_series.value_counts())

EDUC
College Degree         16
Graduate Degree        14
High School Diploma    12
Some College           11
No High School          5
Name: count, dtype: int64


## 9. Assets vs. Debt Scatterplot

In [11]:
# Create dataframe for scatter plot
scatter_df = df_fear[['ASSET', 'DEBT', 'INCOME', 'AGE']].copy()
scatter_df = scatter_df[
    (scatter_df['ASSET'] < scatter_df['ASSET'].quantile(0.98)) &
    (scatter_df['DEBT'] < scatter_df['DEBT'].quantile(0.98))
]

fig = px.scatter(
    scatter_df,
    x='ASSET',
    y='DEBT',
    color='AGE',
    color_continuous_scale='Viridis',
    title='Total Assets vs. Total Debt (colored by Age)',
    labels={'ASSET': 'Total Assets ($)', 'DEBT': 'Total Debt ($)', 'AGE': 'Age'},
    opacity=0.6
)
fig.show()

## 10. Income vs. Net Worth Scatterplot

In [12]:
# Income vs net worth
scatter2_df = df_fear[['INCOME', 'NETWORTH', 'EDUC']].copy()
scatter2_df['Education'] = scatter2_df['EDUC'].map(educ_labels)
scatter2_df = scatter2_df[
    (scatter2_df['INCOME'] < scatter2_df['INCOME'].quantile(0.97)) &
    (scatter2_df['NETWORTH'] > scatter2_df['NETWORTH'].quantile(0.02)) &
    (scatter2_df['NETWORTH'] < scatter2_df['NETWORTH'].quantile(0.97))
]

fig = px.scatter(
    scatter2_df,
    x='INCOME',
    y='NETWORTH',
    color='Education',
    title='Income vs. Net Worth (colored by Education Level)',
    labels={'INCOME': 'Annual Income ($)', 'NETWORTH': 'Net Worth ($)'},
    opacity=0.7
)
fig.show()

## 11. Credit Card Debt Distribution

In [13]:
# Credit card debt — households with CC debt
cc_series = df_fear['CCBAL']
cc_with_debt = cc_series[cc_series > 0]

print(f'Households with CC debt: {len(cc_with_debt):,} ({len(cc_with_debt)/len(df_fear)*100:.1f}%)')
print(f'Mean CC balance:         ${cc_with_debt.mean():,.0f}')
print(f'Median CC balance:       ${cc_with_debt.median():,.0f}')

fig = px.histogram(
    x=cc_with_debt[cc_with_debt < cc_with_debt.quantile(0.99)],
    nbins=40,
    title='Credit Card Balance Distribution (households with CC debt, excl. top 1%)',
    labels={'x': 'Credit Card Balance ($)', 'count': 'Number of Households'},
    color_discrete_sequence=['#AB63FA']
)
fig.update_layout(bargap=0.05)
fig.show()

Households with CC debt: 490 (52.9%)
Mean CC balance:         $9,285
Median CC balance:       $2,782


## 12. Mortgage Debt Histogram

In [14]:
# Mortgage debt for homeowners
mortgage_series = df_fear['MRTHEL']
has_mortgage = mortgage_series[mortgage_series > 0]

print(f'Households with mortgage: {len(has_mortgage):,} ({len(has_mortgage)/len(df_fear)*100:.1f}%)')
print(f'Mean mortgage balance:    ${has_mortgage.mean():,.0f}')
print(f'Median mortgage balance:  ${has_mortgage.median():,.0f}')

fig = px.histogram(
    x=has_mortgage[has_mortgage < has_mortgage.quantile(0.99)],
    nbins=40,
    title='Mortgage Debt Distribution (households with mortgage, excl. top 1%)',
    labels={'x': 'Mortgage Balance ($)', 'count': 'Number of Households'},
    color_discrete_sequence=['#FFA15A']
)
fig.update_layout(bargap=0.05)
fig.show()

Households with mortgage: 245 (26.5%)
Mean mortgage balance:    $362,146
Median mortgage balance:  $173,881


## 13. Correlation Matrix

In [15]:
# Correlation matrix for key financial variables
corr_cols = ['INCOME', 'NETWORTH', 'ASSET', 'DEBT', 'HOUSES',
             'MRTHEL', 'CCBAL', 'RETQLIQ', 'NHNFIN', 'AGE']

corr_matrix = df_fear[corr_cols].corr()

fig = px.imshow(
    corr_matrix,
    title='Correlation Matrix: Key Financial Variables (TURNFEAR == 1)',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    text_auto='.2f'
)
fig.update_layout(width=700, height=700)
fig.show()

## 14. Correlation with TURNFEAR (Full Dataset)

In [16]:
# Correlation of financial variables with TURNFEAR in full dataset
turnfear_corr = df[corr_cols + ['TURNFEAR']].corr()['TURNFEAR'].drop('TURNFEAR').sort_values()

fig = px.bar(
    x=turnfear_corr.values,
    y=turnfear_corr.index,
    orientation='h',
    title='Correlation of Financial Variables with TURNFEAR',
    labels={'x': 'Pearson Correlation', 'y': 'Variable'},
    color=turnfear_corr.values,
    color_continuous_scale='RdBu'
)
fig.add_vline(x=0, line_color='black', line_width=1)
fig.show()

print('\nCorrelation coefficients with TURNFEAR:')
print(turnfear_corr.to_string())


Correlation coefficients with TURNFEAR:
AGE        $-0.22
RETQLIQ    $-0.12
HOUSES     $-0.09
ASSET      $-0.07
NETWORTH   $-0.07
NHNFIN     $-0.05
MRTHEL     $-0.05
INCOME     $-0.04
DEBT       $-0.02
CCBAL       $0.07


## 15. Age vs. Income by Credit Fear Status

In [17]:
# Compare fear vs non-fear groups
comparison_df = df[['AGE', 'INCOME', 'NETWORTH', 'CCBAL', 'DEBT', 'TURNFEAR']].copy()
comparison_df['Credit Status'] = comparison_df['TURNFEAR'].map(
    {1: 'Turned Down / Feared Denial', 0: 'No Credit Issues'}
)

# Filter extremes for visualization
comparison_df = comparison_df[
    comparison_df['INCOME'] < comparison_df['INCOME'].quantile(0.97)
]

fig = px.box(
    comparison_df,
    x='Credit Status',
    y='INCOME',
    color='Credit Status',
    title='Income Distribution: Credit Fear vs. No Credit Issues',
    labels={'INCOME': 'Annual Household Income ($)'},
    color_discrete_map={
        'Turned Down / Feared Denial': '#EF553B',
        'No Credit Issues': '#636EFA'
    }
)
fig.show()

## 16. Summary: Key Findings

In [18]:
# Summary comparison table
fear_group = df[df['TURNFEAR'] == 1]
no_fear_group = df[df['TURNFEAR'] == 0]

summary_comparison = pd.DataFrame({
    'Metric': ['Count', 'Median Income', 'Median Net Worth', 'Median CC Debt',
               'Median Total Debt', 'Median Age', 'Pct with Mortgage'],
    'Turned Down / Feared Denial': [
        f'{len(fear_group):,}',
        f'${fear_group["INCOME"].median():,.0f}',
        f'${fear_group["NETWORTH"].median():,.0f}',
        f'${fear_group["CCBAL"].median():,.0f}',
        f'${fear_group["DEBT"].median():,.0f}',
        f'{fear_group["AGE"].median():.0f} years',
        f'{(fear_group["MRTHEL"] > 0).mean()*100:.1f}%'
    ],
    'No Credit Issues': [
        f'{len(no_fear_group):,}',
        f'${no_fear_group["INCOME"].median():,.0f}',
        f'${no_fear_group["NETWORTH"].median():,.0f}',
        f'${no_fear_group["CCBAL"].median():,.0f}',
        f'${no_fear_group["DEBT"].median():,.0f}',
        f'{no_fear_group["AGE"].median():.0f} years',
        f'{(no_fear_group["MRTHEL"] > 0).mean()*100:.1f}%'
    ]
})

print('=== KEY FINDINGS ===')
print(summary_comparison.to_string(index=False))

=== KEY FINDINGS ===
           Metric Turned Down / Feared Denial No Credit Issues
            Count                         926            4,851
    Median Income                     $47,210         $109,763
 Median Net Worth                     $13,876         $411,982
   Median CC Debt                        $180               $0
Median Total Debt                     $21,457          $36,863
       Median Age                    44 years         56 years
Pct with Mortgage                       26.5%            44.2%
